# Part 10 — Unsupervised biophysical zoning + profiling (decorrelated)

Cluster a **curated, decorrelated** subset of the z-scored stack into biophysical zones, then profile each zone. Engine: `src/zoning.py`.

**Decorrelation (2026-07-14):** clustering the raw 34-band stack resolved only 3 low-expressiveness zones — GO is climatically near-uniform yet the 14 collinear climate bands each carried full Euclidean weight and swamped the soil/relief/hydrology structure. We now cluster on `zoning.ZONING_BANDS` (~15 bands) with **theme-block weighting** (÷√bands-in-theme) then **PCA** (≥90% variance), and **re-select k** by silhouette + Davies–Bouldin + the gap statistic (expect k=4–6).

**Clustering is offline (scikit-learn)**, not EE weka. We fit KMeans in PC space, then classify the full image server-side by projecting bands onto the PCA loadings and taking the **nearest centroid** (linear band-math, label-identical to sklearn).

**Guardrail:** only z-features feed the clusterer; the raw stack, `suit_*` and realized-use fractions ride along **for profiling only** — never as clustering inputs.

**Output:** `zones_present` + `zone_profiles.csv`. **DoD:** k≥4 with defensible internal validity; zones biophysically distinct; profile cards carry a de-meaned suitability signature.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import ee, geemap
import utils, features
project = utils.init()
print('EE initialized; project =', project)
import zoning, external

In [ ]:
aoi = utils.load_aoi(project)
print('AOI area (km^2):', round(aoi.area(1000).divide(1e6).getInfo(), 1))

### Load the z-stack (clustering) + raw stack, suitability, realized use (profiling)
Clustering uses only `zoning.ZONING_BANDS` (the curated decorrelated subset); the realized role fractions ride along so profile cards report each zone's land-use composition.

In [ ]:
z = ee.Image(utils.asset_id(project, 'feature_stack_250m_z'))
raw = ee.Image(utils.asset_id(project, 'feature_stack_250m'))
suit = ee.Image(utils.asset_id(project, 'suit_present'))
realized = external.realized_features(aoi)
band_names = zoning.ZONING_BANDS
frac_bands = [f'rl_{r}_frac' for r in ('soybean','sugarcane','other_crops','pasture','native')]
segs = list(utils.cfg('segments')['segments'])
print(f'{len(band_names)} curated clustering bands; {len(segs)} segments')

### Sample once (aligned): `z_*` curated features + raw + `suit_*` + realized fractions
Pulled to a DataFrame for offline decorrelation + clustering. For very large `n`, swap the `fc_to_df` getInfo for an `Export.table` to Drive.

In [ ]:
sample = zoning.build_sample(z, raw, suit, aoi, band_names, n=15000, seed=42,
                             extra=realized.select(frac_bands))
df = zoning.fc_to_df(sample)
# theme-block-weighted design matrix, then PCA (>=90% variance) to decorrelate
X = zoning.cluster_matrix(df, band_names)          # theme-weighted z-features
pca = zoning.fit_pca(X, var_keep=0.90)
S = pca.transform(X)
print(f'design {X.shape} -> {pca.n_components_} PCs (>=90% var); PC space {S.shape}')

### Re-select k on the decorrelated PC space — silhouette + Davies–Bouldin + gap
Cluster in PC space. Lower Davies–Bouldin and higher silhouette are better; the gap statistic's first `gap(k) >= gap(k+1) - s_{k+1}` is the recommended k. Default picks the max-silhouette k; override `K` if the other criteria argue otherwise (expect 4–6).

In [ ]:
sweep = zoning.kmeans_sweep(S, ks=range(2, 11), seed=42)
gap = zoning.gap_statistic(S, ks=range(2, 11), B=10, seed=42)
swp = sweep.merge(gap, on='k')
print(swp.round(3).to_string(index=False))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
ax[0].plot(swp.k, swp.silhouette, 'o-'); ax[0].set(title='silhouette (max)', xlabel='k')
ax[1].plot(swp.k, swp.davies_bouldin, 'o-'); ax[1].set(title='Davies-Bouldin (min)', xlabel='k')
ax[2].errorbar(swp.k, swp.gap, yerr=swp.s_k, fmt='o-'); ax[2].set(title='gap', xlabel='k')
plt.tight_layout(); plt.show()
K = int(swp.loc[swp.silhouette.idxmax(), 'k'])
print('chosen K =', K, '(review DB + gap before committing)')

### Fit final KMeans in PC space, classify the full image (nearest-centroid in PC space)

In [ ]:
km = zoning.fit_kmeans(S, K, seed=42)
import numpy as np
print('zone sizes (sample):', np.bincount(km.labels_))
# project the z-image onto the PCA loadings server-side, then nearest-centroid
theme_w = zoning.theme_weight_vector(band_names)
pc_img = zoning.pca_project_image(z, band_names, theme_w, pca)
zones = zoning.nearest_centroid_image(pc_img, zoning.pc_names(pca), km.cluster_centers_)
print('zone band:', zones.bandNames().getInfo())

### Profile each zone — feature means, comparative best-use, top features, composition
`comparative_segment` (argmax of zone-mean z-normalized suitability) is the zone's headline label (replaces the ad-hoc distinctive-segment); `top_features` are its most distinctive z-bands; `rl_*_frac` give the realized land-use composition. Profiling uses the sample labels (offline, quota-light; labels match the exported `zones_present`).

In [ ]:
prof = zoning.profile_zones(df, km.labels_, band_names, segs, realized_fracs=frac_bands)
prof.to_csv('zone_profiles.csv', index=False)
cols = ['zone', 'n', 'comparative_segment', 'dominant_segment', 'top_features']
prof[cols + [f'suit_{s}' for s in segs]].round(3)

### Quick look — zone map

In [ ]:
Map = geemap.Map(); Map.centerObject(aoi, 7)
pal = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']
Map.addLayer(zones, {'min': 0, 'max': K - 1, 'palette': pal[:K]}, 'zones')
Map.addLayer(aoi, {}, 'AOI', False)
Map

### Export `zones_present`

In [ ]:
utils.ensure_folder(project)
task = utils.export_image(zones.toByte(), project, 'zones_present', aoi)
print('export', task.status()['description'], '->', task.status()['state'])